# Fill worksite IDs (practice match)

Same rule as `kimedics_sf_mapping.ipynb`:

1. Normalize Kimedics `practice_value` and Salesforce `Job_Client_Job_Id__c` with `norm()`.
2. **Only if** exactly one `Job__c` matches that practice key, treat it as the source of truth.
3. Copy that job’s `Job_Worksite_Location_1__c` into Supabase `sf_worksite_account_id` on **`job_current`** and **`job_content`**.

We do **not** infer worksites from city/state. Optional cleanup clears old bad `sf_worksite_account_id` values before reapplying.

In [13]:
import os, sys, re
from pathlib import Path
from collections import defaultdict

project_root = Path.cwd().resolve()
for _ in range(15):
    if (project_root / "src" / "utils").is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError("Could not locate repo root containing src/utils")

sys.path.insert(0, str(project_root / "src"))
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

# --- Config ---
SCHEMA = "public"  # "public" or "staging"
DRY_RUN = False      # set False to execute cleanup + updates

# Cleanup: wipe prior worksite ids (from old city-level mapping) before reapplying
CLEAR_SF_WORKSITE_ON_JOBS = True   # NULL sf_worksite_account_id on job_current + job_content

if SCHEMA not in ("public", "staging"):
    raise ValueError("SCHEMA must be 'public' or 'staging'")

print("Project root:", project_root)
print("SCHEMA:", SCHEMA, "  DRY_RUN:", DRY_RUN)

SCHEMA: public   DRY_RUN: False


## 1. Load Salesforce jobs + build `practice_norm → Job__c` index

In [14]:
def norm(val):
    """Same as kimedics_sf_mapping.ipynb"""
    s = (val or "").strip().lower()
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"\s*-\s*(closed|closing)\s*$", "", s)
    s = re.sub(r"[,.\-–]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

from utils.salesforce import pull_all_jobs

TOKEN_URL = os.environ.get("SALESFORCE_TOKEN_URL") or "https://proxi.my.salesforce.com"
USE_SANDBOX = os.environ.get("SALESFORCE_USE_SANDBOX", "").lower() in ("1", "true", "yes")
USE_CC = os.environ.get("SALESFORCE_USE_USERNAME_PASSWORD", "").lower() not in ("1", "true", "yes")

sf_jobs = pull_all_jobs(
    consumer_key=os.environ["SALESFORCE_CONSUMER_KEY"],
    consumer_secret=os.environ["SALESFORCE_CONSUMER_SECRET"],
    username=os.environ.get("SALESFORCE_USERNAME") or None,
    password=os.environ.get("SALESFORCE_PASSWORD") or None,
    use_client_credentials=USE_CC,
    token_url=TOKEN_URL,
    security_token=os.environ.get("SALESFORCE_SECURITY_TOKEN") or None,
    use_sandbox=USE_SANDBOX,
)
print(f"{len(sf_jobs)} Salesforce Job__c records")

sf_by_practice = defaultdict(set)
sf_by_id = {}
for r in sf_jobs:
    sid = r["Id"]
    sf_by_id[sid] = r
    p = norm(r.get("Job_Client_Job_Id__c"))
    if p:
        sf_by_practice[p].add(sid)

multi = sum(1 for _k, v in sf_by_practice.items() if len(v) > 1)
print(f"Distinct practice keys: {len(sf_by_practice)} — {multi} keys have multiple SF jobs (will not auto-map)")

4457 Salesforce Job__c records
Distinct practice keys: 812 — 6 keys have multiple SF jobs (will not auto-map)


## 2. Load Kimedics rows + compute 1:1 practice → worksite

In [15]:
from utils.supabase_db import get_conn, get_job_current

with get_conn() as conn:
    kim_rows = get_job_current(conn, limit=None, schema=SCHEMA)
print(f"{len(kim_rows)} job_current rows ({SCHEMA})")

job_to_wid = {}  # job_id -> worksite account id
skipped = {"no_practice": 0, "no_match": 0, "ambiguous": 0}

for kr in kim_rows:
    jid = str(kr.get("job_id") or "").strip()
    if not jid:
        continue
    p = norm(kr.get("practice_value"))
    if not p:
        skipped["no_practice"] += 1
        continue
    hits = sorted(sf_by_practice.get(p, set()))
    if len(hits) == 0:
        skipped["no_match"] += 1
        continue
    if len(hits) > 1:
        skipped["ambiguous"] += 1
        continue
    sf = sf_by_id.get(hits[0]) or {}
    wid = (sf.get("Job_Worksite_Location_1__c") or "").strip()
    if wid:
        job_to_wid[jid] = wid

print(f"1:1 mapped with worksite: {len(job_to_wid)}")
print(f"Skipped: {skipped}")

111 job_current rows (public)
1:1 mapped with worksite: 104
Skipped: {'no_practice': 0, 'no_match': 7, 'ambiguous': 0}


## 3. Cleanup (optional)

Removes stale `sf_worksite_account_id` values on `job_current` and `job_content` for this schema.

In [16]:
def run_cleanup(conn):
    if CLEAR_SF_WORKSITE_ON_JOBS:
        with conn.cursor() as cur:
            cur.execute(f'UPDATE "{SCHEMA}".job_current SET sf_worksite_account_id = NULL;')
            n_cur = cur.rowcount
            cur.execute(f'UPDATE "{SCHEMA}".job_content SET sf_worksite_account_id = NULL;')
            n_jc = cur.rowcount
        print(f"Nulled sf_worksite_account_id: job_current={n_cur}, job_content={n_jc}")

if DRY_RUN:
    print("DRY_RUN: would run cleanup if DRY_RUN=False")
else:
    with get_conn() as conn:
        run_cleanup(conn)
        conn.commit()
    print("Cleanup committed.")

Nulled sf_worksite_account_id: job_current=111, job_content=537
Cleanup committed.


## 4. Apply worksite ids to `job_current` + `job_content`

In [17]:
if DRY_RUN:
    print(f"DRY_RUN: would set sf_worksite_account_id for {len(job_to_wid)} job_id(s)")
    for jid, wid in list(job_to_wid.items())[:5]:
        print(f"  {jid} -> {wid}")
    if len(job_to_wid) > 5:
        print(f"  ... and {len(job_to_wid) - 5} more")
else:
    pairs = list(job_to_wid.items())
    with get_conn() as conn:
        with conn.cursor() as cur:
            for jid, wid in pairs:
                cur.execute(
                    f'UPDATE "{SCHEMA}".job_current SET sf_worksite_account_id = %s WHERE job_id = %s;',
                    (wid, jid),
                )
                cur.execute(
                    f'UPDATE "{SCHEMA}".job_content SET sf_worksite_account_id = %s WHERE job_id = %s;',
                    (wid, jid),
                )
        conn.commit()
    print(f"Updated job_current + job_content for {len(pairs)} job_id(s).")

Updated job_current + job_content for 104 job_id(s).
